# Cleaning Feature Table to Compute FC for MElaS

To compute these MElaS values, a feature table must also include the metadata which contains infected and uninfected samples and at least two different endpoints. This code has an example of 3, 12 and 24 weeks. Change these values to fit your data set. This code will make nested folders of every organ inside a folder by condition. For example, you can have a nested folder where files will be stored in MElaSCL0_100/Cecum where samples were infected with strain CL at 100 concentration. To compute elasticity for each metabolite, natural log fold changes and p-values between consecutive endpoints are calculated. The median peak area values for each metabolite of all infected and uninfected samples at each 3, 12 and 24 weeks for each organ are computed. A value of 5.06 x 10⁻⁷, the minimum value in the dataset, needs to be added to each median value to prevent a zero denominator and log error. Fold change (FC) between median infected over median uninfected at each endpoint is then calculated. To calculate elasticity scores, an initial endpoint of 0 weeks is required so FC at 0 weeks is set to a value of 1. Natural log fold changes between consecutive times are then determined for each metabolite. Lastly, p-values using Mann-Whitney U test to compare the differences of infected individuals between 3 versus 12 weeks and 12 versus 24 weeks for each metabolite are computed. 

For each organ, a folder should contain: 
- ratio_0w.csv
- ratio_3.csv
- ratio_12w.csv
- ratio_24w.csv
- newratio_3v0.txt
- newratio_12v3.txt
- newratio_24v12.txt.

The latter three files will be required to calculate MElaS and contain lnFC and p-values

## Import Packages

In [1]:
import pandas as pd #version 2.2.2
import numpy as np #version 1.26.4
from scipy.stats import mannwhitneyu #version 1.15.2
import os

## Make nested folders
First, make a folder where you want to hold all your MelaS values. Here, MElaSRun is the folder. If you want to calculate MElaS values for multiple organs and conditions such as parasite strain and inoculation concentration, add them in organs and folder_type. If not, leave organ and folder_type blank.

In [4]:
organs=['Cecum', 'Distal_Colon', 'Duodenum', 'Esophagus' , 'Heart_A', 'Heart_B', 'Heart_C', 'Heart_D', 'Ileum', 'Jejunum', 'Proximal_Colon', 'Upper_Stomach'] #list of organs of interest
folder_type=['JR0_1000', 'JR0_100', 'CL0_1000', 'CL0_100'] 
for f in folder_type:
    
    for organ in organs:
        nested_directory=f'/Users/kaylapoirier/MElaSRun/MElaS{f}/{organ}' #location to where folders can be added (here it will be added in MElaSRun folder)

        try:
            os.makedirs(nested_directory)
            print(f"Nested directories '{nested_directory}' created successfully.")
        except FileExistsError:
            print(f"One or more directories in '{nested_directory}' already exist.")
        except PermissionError:
            print(f"Permission denied: Unable to create '{nested_directory}'.")
        except Exception as e:
            print(f"An error occurred: {e}")


One or more directories in '/Users/kaylapoirier/MElaSRun/MElaSJR0_1000/Cecum' already exist.
One or more directories in '/Users/kaylapoirier/MElaSRun/MElaSJR0_1000/Distal_Colon' already exist.
One or more directories in '/Users/kaylapoirier/MElaSRun/MElaSJR0_1000/Duodenum' already exist.
One or more directories in '/Users/kaylapoirier/MElaSRun/MElaSJR0_1000/Esophagus' already exist.
One or more directories in '/Users/kaylapoirier/MElaSRun/MElaSJR0_1000/Heart_A' already exist.
One or more directories in '/Users/kaylapoirier/MElaSRun/MElaSJR0_1000/Heart_B' already exist.
One or more directories in '/Users/kaylapoirier/MElaSRun/MElaSJR0_1000/Heart_C' already exist.
One or more directories in '/Users/kaylapoirier/MElaSRun/MElaSJR0_1000/Heart_D' already exist.
One or more directories in '/Users/kaylapoirier/MElaSRun/MElaSJR0_1000/Ileum' already exist.
One or more directories in '/Users/kaylapoirier/MElaSRun/MElaSJR0_1000/Jejunum' already exist.
One or more directories in '/Users/kaylapoirie

## Import and Clean Data

In [8]:
org_dataset=pd.read_csv('DisTol_mergeddata.csv') #import data that includes feature table and metadata
org_dataset=org_dataset.dropna() #drop any NAs
org_dataset= org_dataset[(org_dataset['ATTRIBUTE_infected_parasites'] != 10000)] #remove 10,000 parasite load
org_dataset= org_dataset[(org_dataset['ATTRIBUTE_organ'] != 'Lower_Stomach_2')] #unreliable data
org_dataset= org_dataset[(org_dataset['ATTRIBUTE_organ'] != 'Lower_Stomach')] #unreliable data
org_dataset= org_dataset[(org_dataset['ATTRIBUTE_groupset'] == 'Sample')] #keep only sample and remove QC, blanks, etc data
org_dataset['ATTRIBUTE_endpoint'] = org_dataset['ATTRIBUTE_endpoint'].astype('category') #set endpoint to category (eg 3w)

/var/folders/6y/_pbszwzs24n9w8r5bvzpvy5h0000gn/T/ipykernel_4959/1187546109.py:1: DtypeWarning: Columns (2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  org_dataset=pd.read_csv('DisTol_mergeddata.csv') #import data that includes feature table and metadata


## Make data Frames by Condition
If you want to calculate MElaS values for multiple organs and conditions such as parasite strain and inoculation concentration, make data frames that are separated by each condition combination. If you didn't have other conditions, you can skip this step but make sure to change your data frame where inoculum concentration equal to 0 is "uninfected" and anything above is "infected."

In [9]:
#make dataframe with specific categories

#Data frame with only JR with 0 vs 1000 paraiste load
JR0_1000=org_dataset[(org_dataset['ATTRIBUTE_parasite_species'] != 'CL_Luc')] #keep JR
JR0_1000.loc[JR0_1000['ATTRIBUTE_infected_parasites'] == 0, 'ATTRIBUTE_infected_parasites'] = "uninfected" #change 0 to uninfected
JR0_1000.loc[JR0_1000['ATTRIBUTE_infected_parasites'] == 1000, 'ATTRIBUTE_infected_parasites'] = "infected" #change 1000 to infected 
JR0_1000=JR0_1000[(JR0_1000['ATTRIBUTE_infected_parasites'] != 100)] #remove 100

#Data frame with only JR with 0 vs 100 paraiste load
JR0_100=org_dataset[(org_dataset['ATTRIBUTE_parasite_species'] != 'CL_Luc')] #keep JR
JR0_100.loc[JR0_100['ATTRIBUTE_infected_parasites'] == 0, 'ATTRIBUTE_infected_parasites'] = "uninfected" #change 0 to uninfected
JR0_100.loc[JR0_100['ATTRIBUTE_infected_parasites'] == 100, 'ATTRIBUTE_infected_parasites'] = "infected" #change 100 to infected
JR0_100=JR0_100[(JR0_100['ATTRIBUTE_infected_parasites'] != 1000)] #remove 1000

#Data frame with only CL with 0 vs 1000 paraiste load
CL0_1000=org_dataset[(org_dataset['ATTRIBUTE_parasite_species'] != 'JR_Luc')] #keep CL
CL0_1000.loc[CL0_1000['ATTRIBUTE_infected_parasites'] == 0, 'ATTRIBUTE_infected_parasites'] = "uninfected" #change 0 to uninfected
CL0_1000.loc[CL0_1000['ATTRIBUTE_infected_parasites'] == 1000, 'ATTRIBUTE_infected_parasites'] = "infected" #change 1000 to infected
CL0_1000=CL0_1000[(CL0_1000['ATTRIBUTE_infected_parasites'] != 100)] #remove 100

#Data frame with only CL with 0 vs 100 paraiste load
CL0_100=org_dataset[(org_dataset['ATTRIBUTE_parasite_species'] != 'JR_Luc')] #keep CL
CL0_100.loc[CL0_100['ATTRIBUTE_infected_parasites'] == 0, 'ATTRIBUTE_infected_parasites'] = "uninfected" #change 0 to uninfected
CL0_100.loc[CL0_100['ATTRIBUTE_infected_parasites'] == 100, 'ATTRIBUTE_infected_parasites'] = "infected" #change 100 to infected
CL0_100=CL0_100[(CL0_100['ATTRIBUTE_infected_parasites'] != 1000)] #remove 1000

/var/folders/6y/_pbszwzs24n9w8r5bvzpvy5h0000gn/T/ipykernel_4959/474186529.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'uninfected' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  JR0_1000.loc[JR0_1000['ATTRIBUTE_infected_parasites'] == 0, 'ATTRIBUTE_infected_parasites'] = "uninfected" #change 0 to uninfected
/var/folders/6y/_pbszwzs24n9w8r5bvzpvy5h0000gn/T/ipykernel_4959/474186529.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'uninfected' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  JR0_100.loc[JR0_100['ATTRIBUTE_infected_parasites'] == 0, 'ATTRIBUTE_infected_parasites'] = "uninfected" #change 0 to uninfected
/var/folders/6y/_pbszwzs24n9w8r5bvzpvy5h0000gn/T/ipykernel_4959/474186529.py:17: FutureWarning: Setting an item of 

## Calculate FC Between Consecutive Times
Calculate lnFC of 3w/0w, lnFC 12w/3w  and lnFC 24w/12w and make a file called newratio_3v0.txt, newratio_12v3.txt and newratio_24v12.txt containing lnFC and p-values

In [10]:
positions = org_dataset['ATTRIBUTE_organ'].unique() #make an array of all organ types
infection_time = org_dataset['ATTRIBUTE_endpoint'].unique()  #make an array of all endpoints

datasets={'JR0_1000': JR0_1000, 'JR0_100': JR0_100, 'CL0_1000': CL0_1000, 'CL0_100': CL0_100} #nested folder

#Calculate fold change of median infected/uninfected at time 3w, 12w and 24w of each organ and produce a file for each organ and time point and save it to individual folders by organ
for name, data in datasets.items():
    
    for organ in positions:
        for time in infection_time:

            filtered_df = data[(data['ATTRIBUTE_organ'] == organ) & (data['ATTRIBUTE_endpoint']==time)] #make dataset by specific organ and endpoint
            filtered_df=filtered_df.drop(['sampleid', 'ATTRIBUTE_endpoint', 'ATTRIBUTE_organ', 'ATTRIBUTE_groupset', 'ATTRIBUTE_parasite_species' ], axis=1) #remove metadata columns except columsn that determine infected or uninfected
            
            infected_df=filtered_df[filtered_df["ATTRIBUTE_infected_parasites"]=='infected'] #makes dataframe of just infected by specific organ and post day infection
            uninfected_df=filtered_df[filtered_df["ATTRIBUTE_infected_parasites"]=='uninfected'] #makes dataframe of just uninfected by specific organ and post day infection
         
            median_df=filtered_df.groupby("ATTRIBUTE_infected_parasites").median(numeric_only=True) #caluclate median of infected and uninfected of all metabolites
            ratio_df=(median_df.loc["infected"]+0.0000005)/(median_df.loc["uninfected"]+0.0000005) #median infected+smallest value /median uninfected+smallest value by each time (eg infected 3w/uninfected 3w) **smallest values was 0.00000506
            ratio_df=pd.DataFrame(ratio_df, columns=['FC'])
            
            ratio_df.index.name = 'ID'

            filename = f'/Users/kaylapoirier/MElaSRun/MElaS{name}/{organ}/ratio_{time}.csv' #make a new file called ratio_3w.csv, ratio_12w.csv or ratio_24w.csv for each organ and save it to individual folders by organ
            ratio_df.to_csv(filename)
            
    #Calculate lnFC of 3w/0w, FC 12w/3w and FC 24w/12w and make a file called newratio_3v0.txt, newratio_12v3.txt and newratio_24v12.txt containing lnFC and p-values
    for organ in positions:
        
        #make dataframe of each FC by week
        df_0 = ratio_df.assign(FC=1) #FC at 0 weeks is set to a value of 1
        df_0_filename = f'/Users/kaylapoirier/MElaSRun/MElaS{name}/{organ}/ratio_0w.csv' 
        df_0.to_csv(df_0_filename) #make a new file called ratio_0w.csv save it to individual folders by organ
        df_3=pd.read_csv(f'/Users/kaylapoirier/MElaSRun/MElaS{name}/{organ}/ratio_3w.csv', index_col=0)
        df_12=pd.read_csv(f'/Users/kaylapoirier/MElaSRun/MElaS{name}/{organ}/ratio_12w.csv', index_col=0)
        df_24=pd.read_csv(f'/Users/kaylapoirier/MElaSRun/MElaS{name}/{organ}/ratio_24w.csv', index_col=0)

        #compute ratio and save as a new file
        ratio3_0= df_3['FC'] / df_0['FC'] #FC of 3w/ 0w
        ratio12_3= df_12['FC'] / df_3['FC'] #FC 12w/FC 3w
        ratio24_12= df_24['FC'] / df_12['FC'] #FC 24w/FC 12w
        
        ratio3_0=np.log(ratio3_0) #ln FC3v0
        ratio3_0=pd.DataFrame(ratio3_0, columns=['FC']) #put FC3v12 into dataframe
        ratio3_0.index.name = 'ID'
        filename3v0 = f'/Users/kaylapoirier/MElaSRun/MElaS{name}/{organ}/newratio_3v0.txt'
        ratio3_0.to_csv(filename3v0, sep='\t') #save FC24v12 into a txt file into desingated organ file

        ratio12_3=np.log(ratio12_3) #ln FC12v3
        ratio12_3=pd.DataFrame(ratio12_3, columns=['FC']) #put FC12v3 into dataframe
        ratio12_3.index.name = 'ID'
        filename12v3 = f'/Users/kaylapoirier/MElaSRun/MElaS{name}/{organ}/newratio_12v3.txt'
        ratio12_3.to_csv(filename12v3, sep='\t') #save FC12v3 into a txt file into desingated organ file

        ratio24_12=np.log(ratio24_12) #ln FC24v12
        ratio24_12=pd.DataFrame(ratio24_12, columns=['FC']) #put FC24v12 into dataframe
        ratio24_12.index.name = 'ID'
        filename24v12 = f'/Users/kaylapoirier/MElaSRun/MElaS{name}/{organ}/newratio_24v12.txt'
        ratio24_12.to_csv(filename24v12, sep='\t') #save FC24v12 into a txt file into desingated organ file
        
        
        #Calcualte p-values comparing infected samples to 12w vs 3w and 24w vs 12w
        filtered_df = data[(data['ATTRIBUTE_organ'] == organ)] #filter original dataset by organ

        infected_df_3 = filtered_df[(filtered_df["ATTRIBUTE_infected_parasites"]=='infected') & (filtered_df['ATTRIBUTE_endpoint']=='3w')] #filter infected and 3w
        infected_df_12 = filtered_df[(filtered_df["ATTRIBUTE_infected_parasites"]=='infected') & (filtered_df['ATTRIBUTE_endpoint']=='12w')] #filter infected and 12w
        infected_df_24 = filtered_df[(filtered_df["ATTRIBUTE_infected_parasites"]=='infected') & (filtered_df['ATTRIBUTE_endpoint']=='24w')] #filter infected and 24w
    
        p_values_12_3={}
        p_values_24_12={}
        for column in data.select_dtypes(include=[np.number]).columns:
            infected_vals_3=infected_df_3[column].dropna() #remove NAs for infected
            infected_vals_12=infected_df_12[column].dropna()
            infected_vals_24=infected_df_24[column].dropna()
            
            stats, p_val_12_3=mannwhitneyu(infected_vals_12, infected_vals_3) #p-values comparing infected of 12 w and 3 w of each metabolite in each organ using Mann Whitney test
            p_values_12_3[column]=p_val_12_3 #save to dictionary
            
            stats, p_val_24_12=mannwhitneyu(infected_vals_24, infected_vals_12) #p-values comparing infected of 24 w and 12 w of each metabolite in each and organ
            p_values_24_12[column]=p_val_24_12 #save to dictionary
            
            
        pval_df_12_3 = pd.DataFrame.from_dict(p_values_12_3, orient='index', columns=['adj.P.Val'])
        result_df_12_3 = ratio12_3.join(pval_df_12_3) #join p values of 12v3w with lnFC of each organ
        result_df_12_3.to_csv(filename12v3, sep='\t')
        
        pval_df_24_12 = pd.DataFrame.from_dict(p_values_24_12, orient='index', columns=['adj.P.Val'])
        result_df_24_12 = ratio24_12.join(pval_df_24_12) #join p values of 24v12w with lnFC of each organ
        result_df_24_12.to_csv(filename24v12, sep='\t')

OSError: Cannot save file into a non-existent directory: '/Users/kaylapoirier/MElaSRun/MElaSCL0_1000/Cecum'